In [ ]:
"""
Adversarial attacks on CLIP fine-tuned classifier with validation set.
"""

import random
import shutil
from pathlib import Path

import torch
import torch.nn as nn
import torchattacks
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import save_image
from tqdm import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModel


# ============================================================
# CẤU HÌNH
# ============================================================
CHECKPOINT_PATH  = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/clip_finetuned_1500_base/best_model.pt")     # <-- checkpoint từ train_clip_classifier.py
VAL_DIR          = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset-short-1500/val")                # <-- ImageFolder structure
OUTPUT_DIR       = Path("./adv_results_clip")

IMG_SIZE         = 224                       # CLIP-base dùng 224
N_SAMPLES        = 50                        # số ảnh sample từ val set để attack
BATCH_SIZE       = 8
EPS              = 8 / 255                   # nhiễu tối đa (Linf)
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
SEED             = 42
SAVE_ADV_IMAGES  = True                      # lưu ảnh adv ra disk
# ============================================================


# ------------------------------------------------------------
# CLIP Classifier (cùng kiến trúc với training script)
# ------------------------------------------------------------
class CLIPClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.vision = CLIPVisionModel.from_pretrained(model_name)
        hidden = self.vision.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, pixel_values):
        feats = self.vision(pixel_values=pixel_values).pooler_output
        return self.head(self.dropout(feats))


# ------------------------------------------------------------
# Wrapper: normalize input [0,1] -> CLIP-normalized bên trong
# ------------------------------------------------------------
class NormalizedCLIP(nn.Module):
    """
    torchattacks attack sẽ sinh adversarial trên range [0, 1] (ảnh pixel gốc).
    CLIP model được train với input đã normalize bằng mean/std của CLIP.

    Wrapper này nhận input [0, 1], tự normalize rồi forward qua CLIP.
    Kết quả: attack tạo nhiễu eps trên pixel space thật (dễ hiểu về mặt ngữ nghĩa),
    nhưng model vẫn nhận input đã normalize đúng như lúc train.
    """
    def __init__(self, clip_model, mean, std):
        super().__init__()
        self.clip_model = clip_model
        # Đăng ký làm buffer để tự move theo device
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        # x: [N, 3, H, W] trong range [0, 1]
        x_norm = (x - self.mean) / self.std
        return self.clip_model(x_norm)


# ------------------------------------------------------------
# Load model & data
# ------------------------------------------------------------
def load_clip_model(ckpt_path: Path):
    """Load CLIP checkpoint và bọc lại để tương thích torchattacks."""
    print(f"Loading CLIP checkpoint từ {ckpt_path}...")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    class_names = ckpt["class_names"]
    model_name  = ckpt["model_name"]

    # Load CLIP classifier
    clip_model = CLIPClassifier(model_name, len(class_names)).to(DEVICE).eval()
    clip_model.load_state_dict(ckpt["model_state"])

    # Lấy mean/std của CLIP từ processor
    processor = CLIPImageProcessor.from_pretrained(model_name)
    mean = processor.image_mean
    std  = processor.image_std

    # Wrap với normalization
    model = NormalizedCLIP(clip_model, mean, std).to(DEVICE).eval()

    print(f"  Model: {model_name}")
    print(f"  Classes ({len(class_names)}): {class_names}")
    print(f"  Normalize mean={mean}, std={std}")
    return model, class_names, (mean, std)


def get_val_loader(val_dir: Path, class_names: list, n_samples: int, batch_size: int):
    """
    Tạo DataLoader từ val folder.
    Transform: chỉ resize + ToTensor (KHÔNG normalize).
    Normalization sẽ được wrapper model tự xử lý.

    class_names từ checkpoint: đảm bảo label index khớp với lúc train.
    """
    tfm = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),  # -> [0, 1]
    ])
    dataset = datasets.ImageFolder(str(val_dir), transform=tfm)

    # Kiểm tra class names khớp với checkpoint
    if list(dataset.classes) != list(class_names):
        print(f"[!] CẢNH BÁO: class order trong val folder khác với checkpoint!")
        print(f"    Val folder : {dataset.classes}")
        print(f"    Checkpoint : {class_names}")
        print(f"    Kết quả có thể sai. Đảm bảo val folder đúng class order.")

    print(f"Val set: {len(dataset)} ảnh, {len(dataset.classes)} class: {dataset.classes}")

    # Sample ngẫu nhiên
    random.seed(SEED)
    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))
    subset = Subset(dataset, indices)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=0)
    return loader, dataset.classes


# ------------------------------------------------------------
# Khởi tạo các attack
# ------------------------------------------------------------
def build_attacks(model: nn.Module) -> dict:
    """
    3 whitebox + 3 blackbox.
    Whitebox: FGSM, PGD, C&W
    Blackbox: Square, Pixle, OnePixel
    """
    attacks = {
        # --- Whitebox ---
        "FGSM":     torchattacks.FGSM(model, eps=EPS),
        "PGD":      torchattacks.PGD(model, eps=EPS, alpha=2/255, steps=20),
        "CW":       torchattacks.CW(model, c=1.0, kappa=0, steps=50, lr=0.01),

        # --- Blackbox (query-based) ---
        "Square":   torchattacks.Square(model, norm="Linf", eps=EPS,
                                        n_queries=500, n_restarts=1, seed=SEED),
        "Pixle":    torchattacks.Pixle(model, x_dimensions=(2, 10),
                                       y_dimensions=(2, 10),
                                       restarts=5, max_iterations=20),
        "OnePixel": torchattacks.OnePixel(model, pixels=5, steps=50, popsize=10),
    }
    return attacks


# ------------------------------------------------------------
# Evaluation loops
# ------------------------------------------------------------
@torch.no_grad()
def evaluate_clean(model, loader):
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total


def run_attack(attack_name, atk, model, loader, class_names, save_dir=None):
    """Chạy 1 attack, trả về (adv_accuracy, ASR)."""
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)

    adv_correct, total = 0, 0
    n_saved = 0

    for batch_idx, (imgs, labels) in enumerate(
            tqdm(loader, desc=attack_name, leave=False)):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # Generate adversarial
        adv = atk(imgs, labels)

        # Eval
        with torch.no_grad():
            preds = model(adv).argmax(1)
        adv_correct += (preds == labels).sum().item()
        total += labels.size(0)

        # Lưu vài ảnh đầu (tối đa 10/attack) để check visual
        if save_dir is not None and n_saved < 10:
            for i in range(imgs.size(0)):
                if n_saved >= 10:
                    break
                true_cls = class_names[labels[i].item()]
                pred_cls = class_names[preds[i].item()]
                tag = "success" if preds[i] != labels[i] else "fail"
                fname = f"{n_saved:02d}_{tag}_{true_cls}→{pred_cls}.png"
                save_image(adv[i].clamp(0, 1).cpu(), save_dir / fname)
                n_saved += 1

    adv_acc = adv_correct / total
    asr = 1.0 - adv_acc
    return adv_acc, asr


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def main():
    print(f"Device: {DEVICE}\n")

    if not CHECKPOINT_PATH.exists():
        print(f"[ERROR] Không tìm thấy checkpoint: {CHECKPOINT_PATH}")
        return
    if not VAL_DIR.is_dir():
        print(f"[ERROR] Val folder không tồn tại: {VAL_DIR}")
        return

    # Setup output
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True)

    # Load model + data
    model, class_names, _ = load_clip_model(CHECKPOINT_PATH)
    loader, _ = get_val_loader(VAL_DIR, class_names, N_SAMPLES, BATCH_SIZE)

    # Clean accuracy
    print("\nĐánh giá clean accuracy...")
    clean_acc = evaluate_clean(model, loader)
    print(f"  Clean accuracy: {clean_acc:.2%}\n")

    # Chạy các attack
    attacks = build_attacks(model)
    results = {}

    print("Chạy các attack:\n")
    for name, atk in attacks.items():
        save_dir = OUTPUT_DIR / name if SAVE_ADV_IMAGES else None
        adv_acc, asr = run_attack(name, atk, model, loader, class_names, save_dir)
        results[name] = (adv_acc, asr)
        print(f"  {name:10s}  adv_acc = {adv_acc:.2%}  |  ASR = {asr:.2%}")

    # Bảng kết quả
    print("\n" + "=" * 60)
    print("KẾT QUẢ TỔNG HỢP (CLIP fine-tuned)")
    print("=" * 60)
    print(f"Clean accuracy: {clean_acc:.2%}")
    print(f"Số ảnh attack : {N_SAMPLES}")
    print(f"Eps (Linf)    : {EPS:.4f}\n")

    print(f"{'Attack':<12} {'Type':<10} {'Adv Acc':<10} {'ASR':<10}")
    print("-" * 45)
    types = {
        "FGSM": "whitebox", "PGD": "whitebox", "CW": "whitebox",
        "Square": "blackbox", "Pixle": "blackbox", "OnePixel": "blackbox",
    }
    for name, (adv_acc, asr) in results.items():
        print(f"{name:<12} {types[name]:<10} {adv_acc:<10.2%} {asr:<10.2%}")

    # Lưu report
    report_path = OUTPUT_DIR / "report.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"Model: CLIP fine-tuned ({CHECKPOINT_PATH})\n")
        f.write(f"Clean accuracy: {clean_acc:.4f}\n")
        f.write(f"N_samples: {N_SAMPLES}, eps: {EPS:.4f}\n\n")
        f.write(f"{'Attack':<12} {'Type':<10} {'AdvAcc':<10} {'ASR':<10}\n")
        for name, (adv_acc, asr) in results.items():
            f.write(f"{name:<12} {types[name]:<10} {adv_acc:<10.4f} {asr:<10.4f}\n")
    print(f"\nReport đã lưu tại: {report_path}")
    print(f"Ảnh adversarial mẫu trong: {OUTPUT_DIR}/<attack_name>/")


if __name__ == "__main__":
    main()

/home/dmin/anaconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda

Loading CLIP checkpoint từ /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/clip_finetuned_1500_base/best_model.pt...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 60125.81it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.embeddings.token_embedding.weight                 | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEX

  Model: openai/clip-vit-base-patch32
  Classes (9): ['Bus', 'MPV', 'SUV', 'Sedan', 'Sports Car', 'Truck', 'Van', 'bicycle', 'motorcycle']
  Normalize mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)
Val set: 4050 ảnh, 9 class: ['Bus', 'MPV', 'SUV', 'Sedan', 'Sports Car', 'Truck', 'Van', 'bicycle', 'motorcycle']

Đánh giá clean accuracy...
  Clean accuracy: 90.00%

Chạy các attack:



  FGSM        adv_acc = 16.00%  |  ASR = 84.00%


  PGD         adv_acc = 0.00%  |  ASR = 100.00%


  CW          adv_acc = 6.00%  |  ASR = 94.00%


  Square      adv_acc = 10.00%  |  ASR = 90.00%


  Pixle       adv_acc = 76.00%  |  ASR = 24.00%


  OnePixel    adv_acc = 84.00%  |  ASR = 16.00%

KẾT QUẢ TỔNG HỢP (CLIP fine-tuned)
Clean accuracy: 90.00%
Số ảnh attack : 50
Eps (Linf)    : 0.0314

Attack       Type       Adv Acc    ASR       
---------------------------------------------
FGSM         whitebox   16.00%     84.00%    
PGD          whitebox   0.00%      100.00%   
CW           whitebox   6.00%      94.00%    
Square       blackbox   10.00%     90.00%    
Pixle        blackbox   76.00%     24.00%    
OnePixel     blackbox   84.00%     16.00%    

Report đã lưu tại: adv_results_clip/report.txt
Ảnh adversarial mẫu trong: adv_results_clip/<attack_name>/


In [ ]:
"""
BONUS TASK: Simulate adversarial attack on full pipeline và analysis

"""

import json
import random
import shutil
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torchattacks
from PIL import Image
from torchvision import transforms
from torchvision.utils import save_image
from tqdm import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModel
from transformers import logging as hf_logging
from ultralytics import YOLO

hf_logging.set_verbosity_error()


# ============================================================
# CẤU HÌNH
# ============================================================
VIDEO_INPUT      = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4") 
DET_MODEL_PATH   = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/runs/detect/train/weights/best.pt") 
CLIP_CHECKPOINT  = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/clip_finetuned_1500_base/best_model.pt")
OUTPUT_DIR       = Path("./adv_pipeline_analysis")

# Attack config
N_FRAMES_SAMPLE  = 300               # số frame sample để test
FRAME_STRIDE     = 50               # lấy 1 frame/50, trải đều video
EPS              = 8 / 255
PGD_STEPS        = 20
PGD_ALPHA        = 2 / 255

# Detector config
DET_CONF_THRESHOLD = 0.4
IOU_MATCH_THRESHOLD = 0.5           # IoU để match bbox clean vs adv (đo drift)

# Classifier config
CLIP_IMG_SIZE    = 224

DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
SEED             = 42
SAVE_SAMPLES     = True             # lưu vài ảnh adv để visualize
# ============================================================


# ------------------------------------------------------------
# CLIP classifier + wrapper normalize
# ------------------------------------------------------------
class CLIPClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.vision = CLIPVisionModel.from_pretrained(model_name)
        hidden = self.vision.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, pixel_values):
        feats = self.vision(pixel_values=pixel_values).pooler_output
        return self.head(self.dropout(feats))


class NormalizedCLIP(nn.Module):
    """Accept input [0,1], normalize rồi forward CLIP. Dùng với torchattacks."""
    def __init__(self, clip_model, mean, std):
        super().__init__()
        self.clip_model = clip_model
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return self.clip_model((x - self.mean) / self.std)


def load_clip(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    class_names = ckpt["class_names"]
    model_name  = ckpt["model_name"]

    clip_model = CLIPClassifier(model_name, len(class_names)).to(DEVICE).eval()
    clip_model.load_state_dict(ckpt["model_state"])

    processor = CLIPImageProcessor.from_pretrained(model_name)
    wrapped = NormalizedCLIP(
        clip_model, processor.image_mean, processor.image_std
    ).to(DEVICE).eval()
    return wrapped, class_names, processor


# ------------------------------------------------------------
# Frame attack: attack toàn frame với classifier target
# ------------------------------------------------------------
def attack_full_frame(frame_bgr, clip_model, attack, yolo_det, img_size=CLIP_IMG_SIZE):
    """
    Attack TOÀN FRAME: resize frame -> [0,1] tensor -> attack -> resize lại.

    Ý tưởng: tạo adversarial frame mà CLIP classifier bị fool khi classify
    chính frame đó. Vì YOLO-det đã bias về "có xe ở đây", ta dùng label = class
    mà CLIP predict frame gốc làm target (untargeted attack).
    """
    h, w = frame_bgr.shape[:2]

    # BGR -> RGB -> tensor [0,1] resize về CLIP_IMG_SIZE
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])
    x = tfm(pil).unsqueeze(0).to(DEVICE)

    # Lấy label ground-truth cho attack (dùng prediction của CLIP trên clean)
    with torch.no_grad():
        label = clip_model(x).argmax(1)

    # Generate adversarial
    x_adv = attack(x, label)

    # Resize lại về kích thước gốc của frame (để YOLO-det xử lý đúng scale)
    x_adv_cpu = x_adv[0].detach().cpu().clamp(0, 1)
    # [C, H, W] -> [H, W, C] uint8
    adv_resized = (x_adv_cpu.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    adv_bgr = cv2.cvtColor(adv_resized, cv2.COLOR_RGB2BGR)
    adv_full_size = cv2.resize(adv_bgr, (w, h), interpolation=cv2.INTER_LINEAR)
    return adv_full_size


# ------------------------------------------------------------
# Crop attack: attack từng crop sau khi YOLO-det đã detect
# ------------------------------------------------------------
def attack_crop(crop_bgr, clip_model, attack, img_size=CLIP_IMG_SIZE):
    """Attack 1 crop -> adversarial crop (cùng kích thước)."""
    h, w = crop_bgr.shape[:2]
    if h < 4 or w < 4:
        return crop_bgr

    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])
    x = tfm(pil).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        label = clip_model(x).argmax(1)
    x_adv = attack(x, label)

    # Convert lại BGR, resize về crop gốc
    adv_np = (x_adv[0].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    adv_bgr = cv2.cvtColor(adv_np, cv2.COLOR_RGB2BGR)
    adv_resized = cv2.resize(adv_bgr, (w, h), interpolation=cv2.INTER_LINEAR)
    return adv_resized


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
def iou(bbox1, bbox2):
    """IoU giữa 2 bbox (x1, y1, x2, y2)."""
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])
    if x2 <= x1 or y2 <= y1:
        return 0.0
    inter = (x2 - x1) * (y2 - y1)
    a1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    a2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    return inter / (a1 + a2 - inter)


def match_bboxes(clean_boxes, adv_boxes, iou_threshold=0.5):
    """
    Match bbox clean <-> adv theo IoU.
    Trả về: (n_matched, n_lost_from_clean, n_new_in_adv, mean_iou)
    """
    if len(clean_boxes) == 0 and len(adv_boxes) == 0:
        return 0, 0, 0, 0.0
    if len(clean_boxes) == 0:
        return 0, 0, len(adv_boxes), 0.0
    if len(adv_boxes) == 0:
        return 0, len(clean_boxes), 0, 0.0

    matched = set()
    ious_matched = []
    for i, cb in enumerate(clean_boxes):
        best_iou, best_j = 0.0, -1
        for j, ab in enumerate(adv_boxes):
            if j in matched:
                continue
            cur = iou(cb, ab)
            if cur > best_iou:
                best_iou, best_j = cur, j
        if best_j >= 0 and best_iou >= iou_threshold:
            matched.add(best_j)
            ious_matched.append(best_iou)

    n_matched = len(ious_matched)
    n_lost = len(clean_boxes) - n_matched
    n_new  = len(adv_boxes) - n_matched
    mean_iou = float(np.mean(ious_matched)) if ious_matched else 0.0
    return n_matched, n_lost, n_new, mean_iou


def classify_crops(clip_model, frame_bgr, boxes):
    """Classify nhiều crop cùng lúc, trả về list (label_idx, conf)."""
    if len(boxes) == 0:
        return []
    h, w = frame_bgr.shape[:2]
    tfm = transforms.Compose([
        transforms.Resize((CLIP_IMG_SIZE, CLIP_IMG_SIZE)),
        transforms.ToTensor(),
    ])
    crops = []
    for (x1, y1, x2, y2) in boxes:
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(w, int(x2)), min(h, int(y2))
        if x2 <= x1 or y2 <= y1:
            continue
        crop = frame_bgr[y1:y2, x1:x2]
        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        crops.append(tfm(Image.fromarray(rgb)))
    if not crops:
        return []
    batch = torch.stack(crops).to(DEVICE)
    with torch.no_grad():
        logits = clip_model(batch)
        probs = torch.softmax(logits, dim=-1)
        conf, pred = probs.max(1)
    return list(zip(pred.cpu().tolist(), conf.cpu().tolist()))


# ------------------------------------------------------------
# Evaluation cho 1 scenario
# ------------------------------------------------------------
def run_scenario(frames, scenario_name, yolo_det, clip_model,
                 attack, attack_target, save_dir=None):
    """
    attack_target: "frame" | "crop" | None (clean baseline)

    Returns dict metrics per-frame và aggregate.
    """
    results = {
        "scenario": scenario_name,
        "per_frame": [],
    }

    for idx, frame in enumerate(tqdm(frames, desc=scenario_name, leave=False)):
        # --- Clean baseline (trên frame gốc) ---
        clean_det = yolo_det.predict(frame, conf=DET_CONF_THRESHOLD,
                                     verbose=False, device=DEVICE)[0]
        clean_boxes = (clean_det.boxes.xyxy.cpu().numpy()
                       if clean_det.boxes is not None else np.zeros((0, 4)))
        clean_confs = (clean_det.boxes.conf.cpu().numpy()
                       if clean_det.boxes is not None else np.zeros(0))
        clean_cls_results = classify_crops(clip_model, frame, clean_boxes.tolist())
        clean_labels = [c[0] for c in clean_cls_results]

        # --- Attack scenario ---
        adv_frame = frame.copy()
        if attack_target == "frame":
            # Attack toàn frame -> chạy detector trên adv_frame
            adv_frame = attack_full_frame(frame, clip_model, attack, yolo_det)
            adv_det = yolo_det.predict(adv_frame, conf=DET_CONF_THRESHOLD,
                                       verbose=False, device=DEVICE)[0]
            adv_boxes = (adv_det.boxes.xyxy.cpu().numpy()
                         if adv_det.boxes is not None else np.zeros((0, 4)))
            adv_confs = (adv_det.boxes.conf.cpu().numpy()
                         if adv_det.boxes is not None else np.zeros(0))
            adv_cls_results = classify_crops(clip_model, adv_frame, adv_boxes.tolist())

        elif attack_target == "crop":
            # Detection trên frame gốc, attack từng crop trước khi classify
            adv_boxes = clean_boxes.copy()
            adv_confs = clean_confs.copy()
            h, w = frame.shape[:2]
            adv_cls_results = []
            for (x1, y1, x2, y2) in adv_boxes:
                x1i, y1i = max(0, int(x1)), max(0, int(y1))
                x2i, y2i = min(w, int(x2)), min(h, int(y2))
                if x2i <= x1i or y2i <= y1i:
                    continue
                crop = frame[y1i:y2i, x1i:x2i]
                adv_crop = attack_crop(crop, clip_model, attack)
                # Classify adv crop
                res = classify_crops(clip_model, adv_crop, [(0, 0, adv_crop.shape[1], adv_crop.shape[0])])
                if res:
                    adv_cls_results.append(res[0])

        else:
            # Clean baseline - không attack
            adv_boxes = clean_boxes
            adv_confs = clean_confs
            adv_cls_results = clean_cls_results

        # --- Metrics ---
        n_matched, n_lost, n_new, mean_iou_val = match_bboxes(
            clean_boxes.tolist(), adv_boxes.tolist(), IOU_MATCH_THRESHOLD)

        # Classification change rate (với bbox matched)
        # Để đơn giản: so sánh theo index (chỉ valid cho crop attack)
        cls_changed = 0
        cls_total = 0
        if attack_target == "crop":
            for i, (clean_res, adv_res) in enumerate(
                    zip(clean_cls_results, adv_cls_results)):
                cls_total += 1
                if clean_res[0] != adv_res[0]:
                    cls_changed += 1

        per_frame = {
            "frame_idx": idx,
            "n_det_clean": len(clean_boxes),
            "n_det_adv":   len(adv_boxes),
            "det_conf_clean_mean": float(clean_confs.mean()) if len(clean_confs) else 0,
            "det_conf_adv_mean":   float(adv_confs.mean())   if len(adv_confs)   else 0,
            "n_matched": n_matched,
            "n_lost":    n_lost,
            "n_new":     n_new,
            "mean_iou":  mean_iou_val,
            "cls_changed": cls_changed,
            "cls_total":   cls_total,
        }
        results["per_frame"].append(per_frame)

        # Save sample adv frame (chỉ vài frame đầu)
        if save_dir is not None and idx < 3:
            save_dir.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(save_dir / f"frame_{idx:02d}_clean.jpg"), frame)
            if attack_target == "frame":
                cv2.imwrite(str(save_dir / f"frame_{idx:02d}_adv.jpg"), adv_frame)
                # Diff ×10 để thấy nhiễu
                diff = cv2.absdiff(frame, adv_frame).astype(np.int32) * 10
                diff = np.clip(diff, 0, 255).astype(np.uint8)
                cv2.imwrite(str(save_dir / f"frame_{idx:02d}_diff_x10.jpg"), diff)

    # Aggregate
    pf = results["per_frame"]
    n_frames = len(pf)
    agg = {
        "n_frames": n_frames,
        "total_det_clean": sum(p["n_det_clean"] for p in pf),
        "total_det_adv":   sum(p["n_det_adv"]   for p in pf),
        "total_matched":   sum(p["n_matched"]   for p in pf),
        "total_lost":      sum(p["n_lost"]      for p in pf),
        "total_new":       sum(p["n_new"]       for p in pf),
        "mean_iou_matched": float(np.mean([p["mean_iou"] for p in pf
                                           if p["mean_iou"] > 0]) or 0),
        "mean_conf_clean": float(np.mean([p["det_conf_clean_mean"] for p in pf
                                          if p["det_conf_clean_mean"] > 0]) or 0),
        "mean_conf_adv":   float(np.mean([p["det_conf_adv_mean"] for p in pf
                                          if p["det_conf_adv_mean"] > 0]) or 0),
        "cls_changed":     sum(p["cls_changed"] for p in pf),
        "cls_total":       sum(p["cls_total"]   for p in pf),
    }
    # Derived
    if agg["total_det_clean"] > 0:
        agg["det_recall"] = agg["total_matched"] / agg["total_det_clean"]
        agg["det_loss_rate"] = agg["total_lost"]  / agg["total_det_clean"]
    else:
        agg["det_recall"] = agg["det_loss_rate"] = 0.0

    if agg["cls_total"] > 0:
        agg["cls_flip_rate"] = agg["cls_changed"] / agg["cls_total"]
    else:
        agg["cls_flip_rate"] = 0.0

    results["aggregate"] = agg
    return results


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def main():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # Validate
    for p, name in [(VIDEO_INPUT, "video"),
                    (DET_MODEL_PATH, "YOLO-det"),
                    (CLIP_CHECKPOINT, "CLIP checkpoint")]:
        if not p.exists():
            print(f"[ERROR] Không tìm thấy {name}: {p}")
            return

    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True)

    print(f"Device: {DEVICE}\n")

    # Load models
    print("Load YOLO-det...")
    yolo_det = YOLO(str(DET_MODEL_PATH))
    print(f"  Det classes: {yolo_det.names}\n")

    print("Load CLIP...")
    clip_model, class_names, _ = load_clip(CLIP_CHECKPOINT)
    print(f"  Classes: {class_names}\n")

    # Sample frames từ video
    cap = cv2.VideoCapture(str(VIDEO_INPUT))
    if not cap.isOpened():
        print(f"[ERROR] Không mở được video: {VIDEO_INPUT}")
        return
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    frame_idx = 0
    while len(frames) < N_FRAMES_SAMPLE:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        frame_idx += FRAME_STRIDE
        if frame_idx >= total:
            break
    cap.release()
    print(f"Sampled {len(frames)} frames từ video ({total} total frames)\n")

    # Khởi tạo attacks
    fgsm = torchattacks.FGSM(clip_model, eps=EPS)
    pgd  = torchattacks.PGD(clip_model, eps=EPS, alpha=PGD_ALPHA, steps=PGD_STEPS)

    # Run 5 scenarios
    scenarios = [
        ("Clean Baseline",       None, None),
        ("Frame Attack + FGSM",  fgsm, "frame"),
        ("Frame Attack + PGD",   pgd,  "frame"),
        ("Crop Attack + FGSM",   fgsm, "crop"),
        ("Crop Attack + PGD",    pgd,  "crop"),
    ]

    all_results = []
    for name, attack, target in scenarios:
        print(f"\n[{name}]")
        save_dir = (OUTPUT_DIR / name.replace(" ", "_").replace("+", "")) \
                   if SAVE_SAMPLES else None
        res = run_scenario(frames, name, yolo_det, clip_model,
                           attack, target, save_dir)
        all_results.append(res)
        agg = res["aggregate"]
        print(f"  detections: clean={agg['total_det_clean']}, adv={agg['total_det_adv']}")
        print(f"  det_recall: {agg['det_recall']:.2%} "
              f"(lost {agg['total_lost']}, new {agg['total_new']})")
        print(f"  mean_iou (matched): {agg['mean_iou_matched']:.4f}")
        print(f"  mean_conf: clean={agg['mean_conf_clean']:.4f}, "
              f"adv={agg['mean_conf_adv']:.4f}")
        if target == "crop":
            print(f"  cls_flip_rate: {agg['cls_flip_rate']:.2%} "
                  f"({agg['cls_changed']}/{agg['cls_total']})")

    # ============================================================
    # TỔNG HỢP & PHÂN TÍCH
    # ============================================================
    print("\n" + "=" * 80)
    print("TỔNG HỢP: Attack có ảnh hưởng đến Object Detection không?")
    print("=" * 80)

    baseline = all_results[0]["aggregate"]
    print(f"\nCLEAN BASELINE:")
    print(f"  Total detections : {baseline['total_det_clean']}")
    print(f"  Mean det conf    : {baseline['mean_conf_clean']:.4f}")

    # Bảng so sánh
    print(f"\n{'Scenario':<25} {'n_det':<8} {'recall':<10} "
          f"{'iou':<8} {'conf':<10} {'cls_flip':<10}")
    print("-" * 75)
    for res in all_results:
        agg = res["aggregate"]
        name = res["scenario"]
        cls_flip = f"{agg['cls_flip_rate']:.2%}" if agg['cls_total'] > 0 else "N/A"
        print(f"{name:<25} {agg['total_det_adv']:<8} "
              f"{agg['det_recall']:<10.2%} "
              f"{agg['mean_iou_matched']:<8.4f} "
              f"{agg['mean_conf_adv']:<10.4f} "
              f"{cls_flip:<10}")

    # Kết luận về transferability
    print("\n" + "=" * 80)
    print("KẾT LUẬN TRANSFERABILITY (Attack CLIP → có ảnh hưởng YOLO-det không?)")
    print("=" * 80)

    frame_pgd = all_results[2]["aggregate"]  # Frame Attack + PGD
    crop_pgd  = all_results[4]["aggregate"]  # Crop Attack + PGD

    det_drop_pct = (1 - frame_pgd["det_recall"]) * 100
    conf_drop = baseline["mean_conf_clean"] - frame_pgd["mean_conf_adv"]

    print(f"\n1. FRAME ATTACK (nhiễu trên frame gốc):")
    print(f"   - Detection recall    : {frame_pgd['det_recall']:.2%} "
          f"(tức {det_drop_pct:.1f}% bbox bị 'mất' hoặc shift)")
    print(f"   - Detection conf giảm : {conf_drop:.4f}")
    print(f"   - Mean IoU (matched)  : {frame_pgd['mean_iou_matched']:.4f} "
          f"({'cao' if frame_pgd['mean_iou_matched'] > 0.8 else 'thấp - bbox drift'})")

    print(f"\n2. CROP ATTACK (nhiễu trên crop sau detection):")
    print(f"   - Classification flip : {crop_pgd['cls_flip_rate']:.2%}")
    print(f"   - Detection không bị ảnh hưởng (attack sau det)")

    # Interpretation
    print(f"\n3. PHÂN TÍCH:")
    if det_drop_pct < 10:
        print("   → Nhiễu target CLIP KHÔNG transfer mạnh sang YOLO-det")
        print("   → YOLO-det robust với perturbation CLIP-specific")
    elif det_drop_pct < 30:
        print("   → Nhiễu có CHÚT transfer sang YOLO-det (recall giảm nhẹ)")
        print("   → Hiện tượng cross-model transferability một phần")
    else:
        print("   → Nhiễu TRANSFER MẠNH sang YOLO-det!")
        print("   → Attack classifier cũng làm hỏng detector (adversarial có tính universal)")

    # Save JSON
    report_path = OUTPUT_DIR / "analysis.json"
    serializable = []
    for res in all_results:
        serializable.append({
            "scenario": res["scenario"],
            "aggregate": res["aggregate"],
        })
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump({
            "config": {
                "n_frames": len(frames),
                "eps": EPS,
                "pgd_steps": PGD_STEPS,
                "iou_threshold": IOU_MATCH_THRESHOLD,
            },
            "scenarios": serializable,
        }, f, indent=2)
    print(f"\n[✓] Kết quả đã lưu: {report_path}")
    if SAVE_SAMPLES:
        print(f"[✓] Ảnh samples tại: {OUTPUT_DIR}/<scenario>/")


if __name__ == "__main__":
    main()

Device: cuda

Load YOLO-det...
  Det classes: {0: 'vehicle'}

Load CLIP...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 81098.57it/s]


  Classes: ['Bus', 'MPV', 'SUV', 'Sedan', 'Sports Car', 'Truck', 'Van', 'bicycle', 'motorcycle']

Sampled 121 frames từ video (6023 total frames)


[Clean Baseline]


  detections: clean=3222, adv=3222
  det_recall: 100.00% (lost 0, new 0)
  mean_iou (matched): 1.0000
  mean_conf: clean=0.7840, adv=0.7840

[Frame Attack + FGSM]


  detections: clean=3222, adv=848
  det_recall: 24.64% (lost 2428, new 54)
  mean_iou (matched): 0.9115
  mean_conf: clean=0.7840, adv=0.6994

[Frame Attack + PGD]


  detections: clean=3222, adv=1032
  det_recall: 29.83% (lost 2261, new 71)
  mean_iou (matched): 0.9116
  mean_conf: clean=0.7840, adv=0.7089

[Crop Attack + FGSM]


  detections: clean=3222, adv=3222
  det_recall: 100.00% (lost 0, new 0)
  mean_iou (matched): 1.0000
  mean_conf: clean=0.7840, adv=0.7840
  cls_flip_rate: 65.08% (2097/3222)

[Crop Attack + PGD]


  detections: clean=3222, adv=3222
  det_recall: 100.00% (lost 0, new 0)
  mean_iou (matched): 1.0000
  mean_conf: clean=0.7840, adv=0.7840
  cls_flip_rate: 59.16% (1906/3222)

TỔNG HỢP: Attack có ảnh hưởng đến Object Detection không?

CLEAN BASELINE:
  Total detections : 3222
  Mean det conf    : 0.7840

Scenario                  n_det    recall     iou      conf       cls_flip  
---------------------------------------------------------------------------
Clean Baseline            3222     100.00%    1.0000   0.7840     N/A       
Frame Attack + FGSM       848      24.64%     0.9115   0.6994     N/A       
Frame Attack + PGD        1032     29.83%     0.9116   0.7089     N/A       
Crop Attack + FGSM        3222     100.00%    1.0000   0.7840     65.08%    
Crop Attack + PGD         3222     100.00%    1.0000   0.7840     59.16%    

KẾT LUẬN TRANSFERABILITY (Attack CLIP → có ảnh hưởng YOLO-det không?)

1. FRAME ATTACK (nhiễu trên frame gốc):
   - Detection recall    : 29.83% (tức 70.2